In [3]:
from pyspark.sql import functions as F

landing_path = "abfss://Fleet_Logistics_Engineering@onelake.dfs.fabric.microsoft.com/Fleet_Logistics_Lakehouse.Lakehouse/Files/Landing"

df_route_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(landing_path + "/routes.csv")
)

StatementMeta(, c3f60c85-91ec-41e6-89bb-e768cf161cf0, 5, Finished, Available, Finished, False)

In [4]:
display(df_route_raw)
df_route_raw.printSchema()

print(f"Source records: {df_route_raw.count()}")

StatementMeta(, c3f60c85-91ec-41e6-89bb-e768cf161cf0, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, aba2f16e-525d-4d24-986e-54369fc3ce5e)

root
 |-- route_id: string (nullable = true)
 |-- origin_city: string (nullable = true)
 |-- origin_state: string (nullable = true)
 |-- destination_city: string (nullable = true)
 |-- destination_state: string (nullable = true)
 |-- typical_distance_miles: integer (nullable = true)
 |-- base_rate_per_mile: double (nullable = true)
 |-- fuel_surcharge_rate: double (nullable = true)
 |-- typical_transit_days: integer (nullable = true)

Source records: 58


In [5]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType
)

routes_path = (
    "abfss://Fleet_Logistics_Engineering@onelake.dfs.fabric.microsoft.com/"
    "Fleet_Logistics_Lakehouse.Lakehouse/Files/Landing/routes.csv"
)

StatementMeta(, c3f60c85-91ec-41e6-89bb-e768cf161cf0, 7, Finished, Available, Finished, False)

In [6]:
route_schema = StructType([
    StructField("route_id", StringType(), True),
    StructField("origin_city", StringType(), True),
    StructField("origin_state", StringType(), True),
    StructField("destination_city", StringType(), True),
    StructField("destination_state", StringType(), True),
    StructField("typical_distance_miles", IntegerType(), True),
    StructField("base_rate_per_mile", DoubleType(), True),
    StructField("fuel_surcharge_rate", DoubleType(), True),
    StructField("typical_transit_days", IntegerType(), True)
])

StatementMeta(, c3f60c85-91ec-41e6-89bb-e768cf161cf0, 8, Finished, Available, Finished, False)

In [7]:
df_routes = (
    spark.read
    .option("header", "true")
    .schema(route_schema)
    .csv(routes_path)
)

display(df_routes)

print(f"Source records: {df_routes.count()}")

StatementMeta(, c3f60c85-91ec-41e6-89bb-e768cf161cf0, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d00dfd9d-8f73-4607-9c29-6446f0c013d2)

Source records: 58


In [9]:
# 1. NULL primary keys
null_route_ids = (
    df_routes
    .filter(F.col("route_id").isNull())
    .count()
)
print(f"NULL route IDs: {null_route_ids}")

StatementMeta(, c3f60c85-91ec-41e6-89bb-e768cf161cf0, 11, Finished, Available, Finished, False)

NULL route IDs: 0


In [10]:
# 2. Duplicate primary keys
duplicate_route_ids = (
    df_routes
    .groupBy("route_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"Duplicate route IDs: {duplicate_route_ids}")

StatementMeta(, c3f60c85-91ec-41e6-89bb-e768cf161cf0, 12, Finished, Available, Finished, False)

Duplicate route IDs: 0


In [11]:
# 3. Invalid / non-positive distance
invalid_distance = (
    df_routes
    .filter(F.col("typical_distance_miles") <= 0)
    .count()
)
print(f"Invalid distance: {invalid_distance}")

StatementMeta(, c3f60c85-91ec-41e6-89bb-e768cf161cf0, 13, Finished, Available, Finished, False)

Invalid distance: 0


In [12]:
# 4. Invalid base rate
invalid_base_rate = (
    df_routes
    .filter(F.col("base_rate_per_mile") <= 0)
    .count()
)
print(f"Invalid base rate: {invalid_base_rate}")

StatementMeta(, c3f60c85-91ec-41e6-89bb-e768cf161cf0, 14, Finished, Available, Finished, False)

Invalid base rate: 0


In [13]:
# 5. Invalid fuel surcharge
invalid_fuel_surcharge = (
    df_routes
    .filter(F.col("fuel_surcharge_rate") < 0)
    .count()
)
print(f"Invalid fuel surcharge: {invalid_fuel_surcharge}")

StatementMeta(, c3f60c85-91ec-41e6-89bb-e768cf161cf0, 15, Finished, Available, Finished, False)

Invalid fuel surcharge: 0


In [14]:
# 6. Invalid transit days
invalid_transit_days = (
    df_routes
    .filter(F.col("typical_transit_days") <= 0)
    .count()
)
print(f"Invalid transit days: {invalid_transit_days}")

StatementMeta(, c3f60c85-91ec-41e6-89bb-e768cf161cf0, 16, Finished, Available, Finished, False)

Invalid transit days: 0


In [15]:
df_routes = (
    df_routes
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.lit("routes.csv"))
)

df_routes.createOrReplaceTempView("routes_source")

print("Route metadata added successfully.")

StatementMeta(, c3f60c85-91ec-41e6-89bb-e768cf161cf0, 17, Finished, Available, Finished, False)

Route metadata added successfully.


In [16]:
spark.sql("""
        CREATE TABLE IF NOT EXISTS bronze_routes (
            route_id STRING,
            origin_city STRING,
            origin_state STRING,
            destination_city STRING,
            destination_state STRING,
            typical_distance_miles INT,
            base_rate_per_mile DOUBLE,
            fuel_surcharge_rate DOUBLE,
            typical_transit_days INT,
            ingestion_timestamp TIMESTAMP,
            source_file STRING
        )
        USING DELTA
    """)

print("bronze_routes created.")

StatementMeta(, c3f60c85-91ec-41e6-89bb-e768cf161cf0, 18, Finished, Available, Finished, False)

bronze_routes created.


In [17]:
result = spark.sql("""
MERGE INTO bronze_routes AS target
USING routes_source AS source

ON target.route_id = source.route_id

WHEN MATCHED THEN
    UPDATE SET
        target.origin_city = source.origin_city,
        target.origin_state = source.origin_state,
        target.destination_city = source.destination_city,
        target.destination_state = source.destination_state,
        target.typical_distance_miles = source.typical_distance_miles,
        target.base_rate_per_mile = source.base_rate_per_mile,
        target.fuel_surcharge_rate = source.fuel_surcharge_rate,
        target.typical_transit_days = source.typical_transit_days,
        target.ingestion_timestamp = source.ingestion_timestamp,
        target.source_file = source.source_file

WHEN NOT MATCHED THEN
    INSERT (
        route_id,
        origin_city,
        origin_state,
        destination_city,
        destination_state,
        typical_distance_miles,
        base_rate_per_mile,
        fuel_surcharge_rate,
        typical_transit_days,
        ingestion_timestamp,
        source_file
    )
    VALUES (
        source.route_id,
        source.origin_city,
        source.origin_state,
        source.destination_city,
        source.destination_state,
        source.typical_distance_miles,
        source.base_rate_per_mile,
        source.fuel_surcharge_rate,
        source.typical_transit_days,
        source.ingestion_timestamp,
        source.source_file
    )
""")

print("Route Bronze MERGE completed successfully.")

StatementMeta(, c3f60c85-91ec-41e6-89bb-e768cf161cf0, 19, Finished, Available, Finished, False)

Route Bronze MERGE completed successfully.


In [18]:
bronze_count = spark.sql("""
    SELECT COUNT(*) AS record_count
    FROM bronze_routes
""").collect()[0]["record_count"]

print(f"Bronze route records: {bronze_count}")

display(
    spark.sql("""
        SELECT *
        FROM bronze_routes
        ORDER BY route_id
        LIMIT 10
    """)
)

StatementMeta(, c3f60c85-91ec-41e6-89bb-e768cf161cf0, 20, Finished, Available, Finished, False)

Bronze route records: 58


SynapseWidget(Synapse.DataFrame, ec6cd100-b3ad-4e4f-bd6d-d60eaf0b4279)